<a href="https://colab.research.google.com/github/BSsangani/How-Data-Quality-Affects-Energy-Demand-Forecasting-in-UK-Smart-Grids/blob/main/step2_validation_and_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2: Walk-Forward Validation and Hyperparameter Tuning

Before I start the real experiment, I need to fix two things: how I test my
models, and how I choose the best settings for RF and LSTM.

Normal k-fold splits the data randomly. That is a problem for time series
data. A model could train on December data and get tested on October data
from the same year. That means the model can see the future before it
predicts the past. This is not correct. So I am using walk-forward
validation instead. The training data always comes before the test data,
in time order.

This week I am doing ARIMA, Linear Regression, and Random Forest. LSTM will
be next week (it is the last section below). I just need to run that one
cell and the save cell again after.

I need `output/clean_demand_2024_2025.csv` from Step 1 before running this.
I also need to turn on GPU (Runtime > Change runtime type) before the LSTM
part, when I get to it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/energy-project/output', exist_ok=True)
if not os.path.exists('output'):
    os.symlink('/content/drive/MyDrive/energy-project/output', 'output')

Mounted at /content/drive


## 1. Installs

In [ ]:
!pip install -q scikit-learn statsmodels

## 2. Load the data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from itertools import product
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

df = pd.read_csv("/content/clean_demand_2024_2025.csv", parse_dates=["datetime"])
series = df["ND"].values.astype(float)
print(f"Loaded {len(series):,} readings.")

Loaded 35,088 readings.


## 3. Walk-forward splitter

Training data always comes before test data, in time order. Each split moves
the window forward a little.

In [ ]:
def walk_forward_splits(n, n_splits=5, min_train_ratio=0.5):
    # gives back train and test parts, train is always earlier in time
    min_train = int(n * min_train_ratio)
    test_size = (n - min_train) // n_splits
    for i in range(n_splits):
        train_end = min_train + i * test_size
        test_end = train_end + test_size
        if test_end > n:
            break
        yield np.arange(0, train_end), np.arange(train_end, test_end)

print("Splitter ready.")

Splitter ready.


## 4. Lag features

RF and LSTM cannot understand time on their own. They just see rows of
numbers. So I give them the past N readings as input, and the next reading
as the answer to predict. I am using 48 lags, which means the past 24 hours.

In [ ]:
def create_lag_features(series, n_lags):
    X, y = [], []
    for i in range(n_lags, len(series)):
        X.append(series[i - n_lags:i])
        y.append(series[i])
    return np.array(X), np.array(y)

N_LAGS = 48
X_all, y_all = create_lag_features(series, N_LAGS)
print(f"Feature matrix shape: {X_all.shape}")

Feature matrix shape: (35040, 48)


In [ ]:
rf_best_params = {"n_estimators": 500, "max_depth": 30}
rf_best_rmse = 402.82
arima_rmse = 486.52
lr_rmse = 389.80

## 5. Tune Random Forest

I am trying 9 combinations of tree count and depth. Each one is tested
across all the walk-forward folds, not just one, so a setting cannot win by
getting lucky a single time.

In [ ]:
RF_PARAM_GRID = {
    "n_estimators": [100, 300, 500],
    "max_depth": [10, 20, 30],
}

def tune_random_forest(X, y, n_splits=3):
    best_rmse = np.inf
    best_params = None
    results = []

    for n_estimators, max_depth in product(RF_PARAM_GRID["n_estimators"], RF_PARAM_GRID["max_depth"]):
        fold_rmses = []
        for train_idx, test_idx in walk_forward_splits(len(X), n_splits=n_splits):
            model = RandomForestRegressor(
                n_estimators=n_estimators, max_depth=max_depth,
                random_state=42, n_jobs=-1
            )
            model.fit(X[train_idx], y[train_idx])
            preds = model.predict(X[test_idx])
            fold_rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))

        avg_rmse = float(np.mean(fold_rmses))
        results.append({"n_estimators": n_estimators, "max_depth": max_depth, "rmse": avg_rmse})
        print(f"  n_estimators={n_estimators:<4} max_depth={max_depth:<4} -> RMSE = {avg_rmse:.2f}")

        if avg_rmse < best_rmse:
            best_rmse = avg_rmse
            best_params = {"n_estimators": n_estimators, "max_depth": max_depth}

    return best_params, best_rmse, results

print("Tuning RF...\n")
rf_best_params, rf_best_rmse, rf_results = tune_random_forest(X_all, y_all)
print(f"\nBest: {rf_best_params}, RMSE {rf_best_rmse:.2f}")

Tuning RF...

  n_estimators=100  max_depth=10   -> RMSE = 425.92
  n_estimators=100  max_depth=20   -> RMSE = 404.71
  n_estimators=100  max_depth=30   -> RMSE = 404.84
  n_estimators=300  max_depth=10   -> RMSE = 424.89
  n_estimators=300  max_depth=20   -> RMSE = 403.48
  n_estimators=300  max_depth=30   -> RMSE = 403.41
  n_estimators=500  max_depth=10   -> RMSE = 424.38
  n_estimators=500  max_depth=20   -> RMSE = 402.97
  n_estimators=500  max_depth=30   -> RMSE = 402.82

Best: {'n_estimators': 500, 'max_depth': 30}, RMSE 402.82


## 6. ARIMA baseline

Important fix here. My first version had ARIMA forecast the entire test
window in one go (thousands of steps ahead), using only its own earlier
guesses instead of real data. RF and Linear Regression get to see the true
past 48 readings every time, so that was not a fair comparison -- it made
ARIMA's error look far worse than it really is.

Now ARIMA predicts one step at a time, and I feed it the real value before
asking for the next prediction. Same setup as the other two models. I am
using order (2,1,2) for now, no search.

In [ ]:
def evaluate_arima(series, order=(2, 1, 2), n_splits=3):
    fold_rmses = []
    for train_idx, test_idx in walk_forward_splits(len(series), n_splits=n_splits):
        train, test = series[train_idx], series[test_idx]
        model = ARIMA(train, order=order).fit()

        preds = []
        for true_val in test:
            # predict just the next point
            preds.append(model.forecast(steps=1)[0])
            # then feed in the real value before predicting the next one
            model = model.append([true_val], refit=False)

        fold_rmses.append(np.sqrt(mean_squared_error(test, preds)))
    return float(np.mean(fold_rmses))

print("Scoring ARIMA (one step at a time)...\n")
arima_rmse = evaluate_arima(series)
print(f"ARIMA(2,1,2) RMSE = {arima_rmse:.2f}")

Scoring ARIMA (one step at a time)...

ARIMA(2,1,2) RMSE = 486.52


## 7. Linear Regression

This is the simplest model in my proposal. It has no settings to tune. I
just train it and test it the same walk-forward way as the other models.

In [ ]:
def evaluate_linear_regression(X, y, n_splits=3):
    fold_rmses = []
    for train_idx, test_idx in walk_forward_splits(len(X), n_splits=n_splits):
        model = LinearRegression()
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        fold_rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))
    return float(np.mean(fold_rmses))

print("Scoring Linear Regression...\n")
lr_rmse = evaluate_linear_regression(X_all, y_all)
print(f"Linear Regression RMSE = {lr_rmse:.2f}")

Scoring Linear Regression...

Linear Regression RMSE = 389.80


## 8. Save the winning settings

This cell works even if I have not run the LSTM part yet. If LSTM is not
done, it just writes "pending". After I do LSTM, I will run this cell again
to update the file.

In [ ]:
import os
os.makedirs("output", exist_ok=True)

try:
    lstm_result = lstm_best_params
    lstm_rmse_result = lstm_best_rmse
except NameError:
    lstm_result = "pending"
    lstm_rmse_result = None

best_hyperparameters = {
    "random_forest": rf_best_params,
    "linear_regression": "no hyperparameters to tune",
    "arima_order": [2, 1, 2],
    "lstm": lstm_result,
    "validation_strategy": "walk-forward (rolling-origin)",
    "baseline_rmse": {
        "random_forest": rf_best_rmse,
        "linear_regression": lr_rmse,
        "arima": arima_rmse,
        "lstm": lstm_rmse_result,
    },
}

with open("output/best_hyperparameters.json", "w") as f:
    json.dump(best_hyperparameters, f, indent=2)

print("Saved output/best_hyperparameters.json\n")
print(json.dumps(best_hyperparameters, indent=2))

Saved output/best_hyperparameters.json

{
  "random_forest": {
    "n_estimators": 500,
    "max_depth": 30
  },
  "linear_regression": "no hyperparameters to tune",
  "arima_order": [
    2,
    1,
    2
  ],
  "lstm": {
    "units": 64,
    "dropout": 0.1
  },
  "validation_strategy": "walk-forward (rolling-origin)",
  "baseline_rmse": {
    "random_forest": 402.82,
    "linear_regression": 389.8,
    "arima": 486.52,
    "lstm": 445.8833322056434
  }
}


## 9. Tune LSTM (do this next week)

This grid is smaller, only 4 combinations, because LSTM takes much longer to
train than RF. GPU makes a real difference here. After this cell finishes, I
go back to cell 8 (save) and run it again to add LSTM to the results.

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

LSTM_PARAM_GRID = {
    "units": [32, 64],
    "dropout": [0.1, 0.2],
}

def create_lstm_sequences(series, n_lags):
    X, y = create_lag_features(series, n_lags)
    X = X.reshape(-1, n_lags, 1)  # Keras wants [samples, timesteps, features]
    return X, y

def build_lstm(units, dropout, n_lags):
    model = Sequential([
        LSTM(units, input_shape=(n_lags, 1)),
        Dropout(dropout),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def tune_lstm(series, n_lags=N_LAGS, n_splits=2, epochs=5):
    X, y = create_lstm_sequences(series, n_lags)

    # make the numbers smaller so the model trains better -- raw MW values
    # are too big for the network to handle well
    mean, std = y.mean(), y.std()
    X_norm = (X - mean) / std
    y_norm = (y - mean) / std

    best_rmse = np.inf
    best_params = None
    results = []

    for units, dropout in product(LSTM_PARAM_GRID["units"], LSTM_PARAM_GRID["dropout"]):
        fold_rmses = []
        for train_idx, test_idx in walk_forward_splits(len(X_norm), n_splits=n_splits):
            model = build_lstm(units, dropout, n_lags)
            model.fit(X_norm[train_idx], y_norm[train_idx],
                      epochs=epochs, batch_size=64, verbose=0)
            preds_norm = model.predict(X_norm[test_idx], verbose=0).flatten()
            preds = preds_norm * std + mean  # back to real MW numbers
            fold_rmses.append(np.sqrt(mean_squared_error(y[test_idx], preds)))

        avg_rmse = float(np.mean(fold_rmses))
        results.append({"units": units, "dropout": dropout, "rmse": avg_rmse})
        print(f"  units={units:<4} dropout={dropout:<4} -> RMSE = {avg_rmse:.2f}")

        if avg_rmse < best_rmse:
            best_rmse = avg_rmse
            best_params = {"units": units, "dropout": dropout}

    return best_params, best_rmse, results

print("Tuning LSTM...\n")
lstm_best_params, lstm_best_rmse, lstm_results = tune_lstm(series)
print(f"\nBest: {lstm_best_params}, RMSE {lstm_best_rmse:.2f}")





Tuning LSTM...

  units=32   dropout=0.1  -> RMSE = 563.11
  units=32   dropout=0.2  -> RMSE = 516.23
  units=64   dropout=0.1  -> RMSE = 445.88
  units=64   dropout=0.2  -> RMSE = 458.47

Best: {'units': 64, 'dropout': 0.1}, RMSE 445.88


## 10. Download

Click the folder icon on the left, open `output`, and download
`best_hyperparameters.json`. Then upload it to the OneDrive `Software`
folder, next to the Step 1 files.